In [1]:
import random
import numpy as np

In [40]:
class CrossEntropyCost(object):
    @staticmethod
    def fn(a, y):
        return np.sum(np.nan_to_num(-(y*np.log(a) + (1-y)*np.log(1-a))))

    @staticmethod
    def delta(z, a, y):
        return (a-y)

class QudraticCost(object):
    @staticmethod
    def fn(a, y):
        return 0.5 * np.linalg.norm(a-y)**2

    @staticmethod
    def delta(z, a, y):
        return (a - y)*sigmoid_prime(z)


def vectorized_result(j):
    e = np.zeros((10, 1))
    e[j] = 1.0
    return e

def sigmoid_prime(z):
    return sigmoid(z)*(1-sigmoid(z))

def sigmoid(z):
    return 1.0/(1.0+np.exp(-z))

class Network(object):
    def __init__(self, sizes, cost=CrossEntropyCost):
        self.n_layers = len(sizes)
        self.sizes = sizes
        self.cost = cost
        self.default_weight_initializer()

    def default_weight_initializer(self):
        self.weights = [np.random.randn(y, x)/np.sqrt(x) for x,y in zip(self.sizes[:-1], self.sizes[1:])]
        self.biases = [np.random.randn(y, 1) for y in self.sizes[1:]]

    def large_weight_initializer(self):
        self.weights = [np.random.randn(y, x) for x,y in zip(self.sizes[:-1], self.sizes[1:])]
        self.biases = [np.random.randn(y, 1) for y in self.sizes[1:]]

    def feedforward(self, a):
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a) + b)
        return a

    def SGD(self, training_data, epochs, mini_batch_size, eta, lmbda=0.0, evaluation_data=None,
            monitor_evaluation_cost=False,
            monitor_evaluation_accuracy=False,
            monitor_training_cost=False,
            monitor_training_accuracy=False,
            no_improvement_in=None):
        training_data = list(training_data)
        if evaluation_data: n_data = len(evaluation_data)
        n = len(training_data)
        evaluation_cost, evaluation_accuracy = [], []
        training_cost, training_accuracy = [], []
        best_accuracy = -1
        best_epoch = 0
        eta0 = eta
        for j in range(epochs):
            random.shuffle(training_data)
            mini_batches = [training_data[k:k+mini_batch_size] for k in range(0, n, mini_batch_size)]
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta, lmbda, len(training_data))
            print(f"Epoch {j} training complete")
            if monitor_training_cost:
                cost = self.total_cost(training_data, lmbda)
                training_cost.append(cost)
                print(f"Cost on training data: {cost}")
            if monitor_training_accuracy:
                accuracy = self.accuracy(training_data, convert=True)
                training_accuracy.append(accuracy)
                print(f"Accuracy on training data: {accuracy}/{n}")
            if monitor_evaluation_cost:
                cost = self.total_cost(evaluation_data, lmbda, convert=True)
                evaluation_cost.append(cost)
                print(f"Cost on evaluation data: {cost}")
            if monitor_evaluation_accuracy:
                accuracy = self.accuracy(evaluation_data)
                evaluation_accuracy.append(accuracy)
                print(f"Accuracy on evaluation data: {accuracy}/{n_data}")
            # if no_improvement_in is not None:
            #     accuracy = self.accuracy(evaluation_data)
            #     if accuracy > best_accuracy:
            #         best_accuracy = accuracy
            #         best_epoch = j
            #     elif j-best_epoch >= no_improvement_in:
            #         print(f"Early stopping at epoch {j}")
            #         break
            accuracy = self.accuracy(evaluation_data)
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_epoch = j
            elif j - best_epoch >= 5:
                eta /= 2
                if eta < eta0/128:
                    print(f"Early stopping at epoch {j}")
                    break
                best_accuracy, best_epoch = accuracy, j 
        print(f"Learning rate: {eta}")
        return evaluation_cost, evaluation_accuracy, training_cost, training_accuracy

    def update_mini_batch(self, mini_batch, eta, lmbda, n):
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]

        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)
            nabla_b = [nb + dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw + dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]
        self.weights = [(1-eta*lmbda/n)*w - (eta/len(mini_batch))*nw for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b - (eta/len(mini_batch))*nb for b, nb in zip(self.biases, nabla_b)]

    def backprop(self, x, y):
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        zs = []
        activations = [x]
        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activations[-1]) + b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)
        delta = (self.cost).delta(zs[-1], activations[-1], y)
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].transpose())
        for l in range(2, self.n_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            delta = np.dot(self.weights[-l+1].transpose(), delta) * sp
            nabla_b[-l] = delta
            nabla_w[-l] = np.dot(delta, activations[-l-1].transpose())
        return (nabla_b, nabla_w)

    def accuracy(self, data, convert=False):
        if convert:
            results = [(np.argmax(self.feedforward(x)), np.argmax(y)) for (x,y) in data]
        else:
            results = [(np.argmax(self.feedforward(x)), y) for (x, y) in data]
        return sum(int(x==y) for (x, y) in results)

    def total_cost(self, data, lmbda, convert=False):
        cost = 0.0
        for x, y in data:
            a = self.feedforward(x)
            if convert: y = vectorized_result(y)
            cost += self.cost.fn(a, y)/len(data)
        cost += 0.5*(lmbda/len(data))*sum(np.linalg.norm(w)**2 for w in self.weights)
        return cost

In [25]:
import sys
sys.path.append('/Users/michael/ml-course-inno/src')
sys.path.append('/Users/michael/ml-course-inno/data')
import mnist_loader

In [26]:
training_data, validation_data, test_data = mnist_loader.load_data_wrapper()
mlp = Network([784, 30, 10], cost=CrossEntropyCost)
evaluation_cost, evaluation_accuracy, training_cost, training_accuracy = mlp.SGD(training_data, 30, 10, 10.0, lmbda=5.0, evaluation_data=validation_data, 
                                                                              monitor_evaluation_cost=True,
                                                                              monitor_evaluation_accuracy=True,
                                                                              monitor_training_cost=True,
                                                                              monitor_training_accuracy=True)


Epoch 0 training complete
Cost on training data: 0.47670136760174
Accuracy on training data: 46990/50000
Cost on evaluation data: 0.7865600804381125
Accuracy on evaluation data: 9388/10000
Epoch 1 training complete
Cost on training data: 0.4344955835257004
Accuracy on training data: 47589/50000
Cost on evaluation data: 0.8378166718069207
Accuracy on evaluation data: 9509/10000
Epoch 2 training complete
Cost on training data: 0.42090074276102385
Accuracy on training data: 47785/50000
Cost on evaluation data: 0.8844694295922622
Accuracy on evaluation data: 9498/10000
Epoch 3 training complete
Cost on training data: 0.42442536896554484
Accuracy on training data: 47752/50000
Cost on evaluation data: 0.9182768686140536
Accuracy on evaluation data: 9511/10000
Epoch 4 training complete
Cost on training data: 0.3901034846464939
Accuracy on training data: 48127/50000
Cost on evaluation data: 0.9030339529031516
Accuracy on evaluation data: 9549/10000
Epoch 5 training complete
Cost on training da

In [50]:
dumb_net = Network([784, 30, 10], cost=CrossEntropyCost) #not dumb anymore actually :) 
evaluation_cost, evaluation_accuracy, training_cost, training_accuracy = dumb_net.SGD(training_data, 30, 10, 0.1, lmbda=10.0, evaluation_data=validation_data, 
                                                                              monitor_evaluation_cost=True,
                                                                              monitor_evaluation_accuracy=True,
                                                                              monitor_training_cost=True,
                                                                              monitor_training_accuracy=True,
                                                                              no_improvement_in = 5)

Epoch 0 training complete
Cost on training data: 0.6936393796990503
Accuracy on training data: 45320/50000
Cost on evaluation data: 0.8221091889371566
Accuracy on evaluation data: 9157/10000
Epoch 1 training complete
Cost on training data: 0.5847189659580899
Accuracy on training data: 46149/50000
Cost on evaluation data: 0.7929957455460388
Accuracy on evaluation data: 9285/10000
Epoch 2 training complete
Cost on training data: 0.538586390362944
Accuracy on training data: 46568/50000
Cost on evaluation data: 0.7969337092174122
Accuracy on evaluation data: 9352/10000
Epoch 3 training complete
Cost on training data: 0.506811799136626
Accuracy on training data: 46956/50000
Cost on evaluation data: 0.809664083251709
Accuracy on evaluation data: 9409/10000
Epoch 4 training complete
Cost on training data: 0.4886266351276043
Accuracy on training data: 47111/50000
Cost on evaluation data: 0.8238357585163293
Accuracy on evaluation data: 9445/10000
Epoch 5 training complete
Cost on training data: